In [ ]:
%pip install sqlalchemy pymysql lightgbm xgboost streamlit python-dotenv

In [ ]:
from sklearn.datasets import fetch_openml

freq = fetch_openml(data_id=41214, as_frame=True).frame
sev = fetch_openml(data_id=41215, as_frame=True).frame

freq.to_csv("data/raw/freMTPL2freq.csv", index=False)
sev.to_csv("data/raw/freMTPL2sev.csv", index=False)

print("policies:", freq.shape)
print("claims:", sev.shape)

In [3]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, URL

load_dotenv(".env", override=True)

url = URL.create(
    "mysql+pymysql",
    username=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    host=os.environ["DB_HOST"],
    port=int(os.environ["DB_PORT"]),
    database=os.environ["DB_NAME"],
)
engine = create_engine(url)

with engine.connect() as conn:
    print(conn.execute(text("SELECT VERSION()")).scalar())

8.0.46


In [4]:
# Step 1: read the raw CSVs and apply mechanical fixes only (run sql/01_schema_raw.sql first)
import pandas as pd

freq = pd.read_csv("data/raw/freMTPL2freq.csv")
sev = pd.read_csv("data/raw/freMTPL2sev.csv")

for df in (freq, sev):
    for col in df.select_dtypes("object"):
        df[col] = df[col].str.strip("'")

freq = freq.rename(columns={"IDpol": "id_pol", "ClaimNb": "claim_nb", "Exposure": "exposure",
    "Area": "area", "VehPower": "veh_power", "VehAge": "veh_age", "DrivAge": "driv_age",
    "BonusMalus": "bonus_malus", "VehBrand": "veh_brand", "VehGas": "veh_gas",
    "Density": "density", "Region": "region"})
sev = sev.rename(columns={"IDpol": "id_pol", "ClaimAmount": "claim_amount"})
freq["id_pol"] = freq["id_pol"].astype(int)
sev["id_pol"] = sev["id_pol"].astype(int)

print("policies:", freq.shape)
print("claims:", sev.shape)

policies: (678013, 12)
claims: (26639, 2)


In [5]:
# Step 1: append into the raw tables (claim_id is filled by AUTO_INCREMENT)
freq.to_sql("policies_raw", engine, if_exists="append", index=False, chunksize=50_000)
sev.to_sql("claims_raw", engine, if_exists="append", index=False, chunksize=50_000)

26639

In [6]:
# Step 1: reconcile MySQL against the CSVs
with engine.connect() as conn:
    n_policies = conn.execute(text("SELECT COUNT(*) FROM policies_raw")).scalar()
    n_claims = conn.execute(text("SELECT COUNT(*) FROM claims_raw")).scalar()
    total_amount = conn.execute(text("SELECT SUM(claim_amount) FROM claims_raw")).scalar()

print("policies:", n_policies, "vs CSV", len(freq))
print("claims:", n_claims, "vs CSV", len(sev))
print("claim_amount total:", float(total_amount), "vs CSV", round(sev["claim_amount"].sum(), 2))

assert n_policies == len(freq)
assert n_claims == len(sev)
assert abs(float(total_amount) - sev["claim_amount"].sum()) < 1

policies: 678013 vs CSV 678013
claims: 26639 vs CSV 26639
claim_amount total: 60697930.68 vs CSV 60697930.68


In [4]:
import pandas as pd

# Overview of the two raw tables as loaded in MySQL
policies_raw = pd.read_sql("SELECT * FROM policies_raw", engine)
claims_raw = pd.read_sql("SELECT * FROM claims_raw", engine)

for name, df in (("policies_raw", policies_raw), ("claims_raw", claims_raw)):
    print(f"=== {name}: {df.shape[0]:,} rows x {df.shape[1]} columns ===")
    display(df.head(10))
    df.info()
    display(df.describe(include="all").T)

=== policies_raw: 678,013 rows x 12 columns ===


,id_pol,claim_nb,exposure,area,veh_power,veh_age,driv_age,bonus_malus,veh_brand,veh_gas,density,region
0,1,1,0.10,D,5,0,55,50,B12,Regular,1217,R82
1,3,1,0.77,D,5,0,55,50,B12,Regular,1217,R82
2,5,1,0.75,B,6,2,52,50,B12,Diesel,54,R22
3,10,1,0.09,B,7,0,46,50,B12,Diesel,76,R72
4,11,1,0.84,B,7,0,46,50,B12,Diesel,76,R72
5,13,1,0.52,E,6,2,38,50,B12,Regular,3003,R31
6,15,1,0.45,E,6,2,38,50,B12,Regular,3003,R31
7,17,1,0.27,C,7,0,33,68,B12,Diesel,137,R91
8,18,1,0.71,C,7,0,33,68,B12,Diesel,137,R91
9,21,1,0.15,B,7,0,41,50,B12,Diesel,60,R52


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 678013 entries, 0 to 678012
Data columns (total 12 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   id_pol       678013 non-null  int64  
 1   claim_nb     678013 non-null  int64  
 2   exposure     678013 non-null  float64
 3   area         678013 non-null  object 
 4   veh_power    678013 non-null  int64  
 5   veh_age      678013 non-null  int64  
 6   driv_age     678013 non-null  int64  
 7   bonus_malus  678013 non-null  int64  
 8   veh_brand    678013 non-null  object 
 9   veh_gas      678013 non-null  object 
 10  density      678013 non-null  int64  
 11  region       678013 non-null  object 
dtypes: float64(1), int64(7), object(4)
memory usage: 62.1+ MB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id_pol,678013.0,NaN,NaN,NaN,2621856.921071,1641782.752655,1.0,1157951.0,2272152.0,4046274.0,6114330.0
claim_nb,678013.0,NaN,NaN,NaN,0.053247,0.240117,0.0,0.0,0.0,0.0,16.0
exposure,678013.0,NaN,NaN,NaN,0.52875,0.364442,0.0027,0.18,0.49,0.99,2.01
area,678013,6,C,191880,NaN,NaN,NaN,NaN,NaN,NaN,NaN
veh_power,678013.0,NaN,NaN,NaN,6.454631,2.050906,4.0,5.0,6.0,7.0,15.0
veh_age,678013.0,NaN,NaN,NaN,7.044265,5.666232,0.0,2.0,6.0,11.0,100.0
driv_age,678013.0,NaN,NaN,NaN,45.499122,14.137444,18.0,34.0,44.0,55.0,100.0
bonus_malus,678013.0,NaN,NaN,NaN,59.761502,15.636658,50.0,50.0,50.0,64.0,230.0
veh_brand,678013,11,B12,166024,NaN,NaN,NaN,NaN,NaN,NaN,NaN
veh_gas,678013,2,Regular,345877,NaN,NaN,NaN,NaN,NaN,NaN,NaN


=== claims_raw: 26,639 rows x 3 columns ===


,claim_id,id_pol,claim_amount
0,1,1552,995.20
1,2,1010996,1128.12
2,3,4024277,1851.11
3,4,4007252,1204.00
4,5,4046424,1204.00
5,6,4073956,1204.00
6,7,4012173,1204.00
7,8,4020812,54942.62
8,9,4020812,7620.00
9,10,4074074,1204.00


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26639 entries, 0 to 26638
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   claim_id      26639 non-null  int64  
 1   id_pol        26639 non-null  int64  
 2   claim_amount  26639 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 624.5 KB


,count,mean,std,min,25%,50%,75%,max
claim_id,26639.0,1.332000e+04,7.690161e+03,1.0,6660.50,13320.0,19979.50,26639.00
id_pol,26639.0,2.279864e+06,1.577202e+06,139.0,1087642.50,2137413.0,3180162.00,6113971.00
claim_amount,26639.0,2.278536e+03,2.929748e+04,1.0,686.81,1172.0,1228.08,4075400.56
